# VEGFR2 docking (PDB 3VHE) - Abhi - Tryptanthrin anticancer project
Follows the standardised docking workflow: AutoDock Vina 1.2.7, exhaustiveness 32, 9 modes, seed 42, Meeko receptor/ligand preparation.

**Run the cells strictly in order, top to bottom (CELL 1 -> CELL 13).** If a cell says "Run the earlier cells first", scroll up and run the missing one.
Only two cells need your attention: CELL 4 (upload two CSVs) and CELL 5 (check the structure summary). Do not continue past CELL 5/8 if something looks wrong.

Results are computational predictions, not experimentally confirmed activity.

## CELL 1 - Install packages (about 2-3 minutes)

In [ ]:
# CELL 1
!pip install -q vina meeko rdkit gemmi py3Dmol pdbfixer openmm pandas numpy scipy requests
print("install done")

## CELL 2 - Settings (same Vina settings for all targets - do not change)

In [ ]:
# CELL 2
import os
TARGET = "VEGFR2"
PDB_ID = "3VHE"          # starting candidate from the workflow - verified in CELL 5
CHAIN = "A"              # chain used for docking (check CELL 5 output)
LIG_RESNAME = None       # None = auto-pick the crystal ligand in CELL 5; or set e.g. "XYZ"
KEEP_HET = []            # cofactors/metals to keep (normally none for a kinase domain)

EXHAUSTIVENESS = 32
N_MODES = 9
SEED = 42
BOX_EXTRA = 10.0         # box edge = ligand extent + this (A), per axis
BOX_MIN = 22.0           # minimum box edge (A)
RMSD_CUTOFF = 2.0        # redocking pass criterion (A)
FORCE_CONTINUE = False   # only set True if you have investigated a failed redocking

BASE = f"{TARGET}_{PDB_ID}"
D_IN, D_REC, D_REF = f"{BASE}/00_inputs", f"{BASE}/01_receptor", f"{BASE}/02_reference_ligand_redocking"
D_LIG, D_OUT, D_AN, D_FIN = f"{BASE}/03_ligands", f"{BASE}/04_docking_output", f"{BASE}/05_interaction_analysis", f"{BASE}/06_final"
for d in [D_IN, D_REC, D_REF, D_LIG, D_OUT, D_AN, D_FIN]:
    os.makedirs(d, exist_ok=True)
print("folders ready under", BASE)

## CELL 3 - Helper functions (runs instantly, nothing to check)

In [ ]:
import os, re, json, glob, subprocess, warnings
import numpy as np, pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, rdMolAlign, Descriptors
from meeko import MoleculePreparation, PDBQTWriterLegacy, PDBQTMolecule, RDKitMolCreate
from vina import Vina
RDLogger.DisableLog("rdApp.*")

def require(*names):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(f"Run the earlier cells first - missing: {missing}")

def mol_to_pdbqt(mol_h):
    """RDKit mol (with explicit H and 3D coords) -> Meeko PDBQT string."""
    setups = MoleculePreparation().prepare(mol_h)
    pdbqt, ok, err = PDBQTWriterLegacy.write_string(setups[0])
    if not ok:
        raise ValueError(f"Meeko failed: {err}")
    return pdbqt

def prepare_ligand(smiles, seed=42):
    """SMILES -> (3D mol with H, PDBQT string, MMFF status). ETKDGv3 + MMFF as in the workflow."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("invalid SMILES")
    mol = Chem.AddHs(mol)
    ps = AllChem.ETKDGv3(); ps.randomSeed = seed
    if AllChem.EmbedMolecule(mol, ps) != 0:
        ps.useRandomCoords = True
        if AllChem.EmbedMolecule(mol, ps) != 0:
            raise ValueError("3D embedding failed")
    status = AllChem.MMFFOptimizeMolecule(mol, maxIters=5000)   # 0 = converged, 1 = not converged, -1 = no params
    return mol, mol_to_pdbqt(mol), status

def run_vina(pdbqt_str, receptor_pdbqt, center, size, exhaustiveness=32, n_modes=9, seed=42):
    v = Vina(sf_name="vina", seed=seed, verbosity=0)
    v.set_receptor(receptor_pdbqt)
    v.set_ligand_from_string(pdbqt_str)
    v.compute_vina_maps(center=list(map(float, center)), box_size=list(map(float, size)))
    v.dock(exhaustiveness=exhaustiveness, n_poses=n_modes)
    energies = v.energies(n_poses=n_modes)          # columns: total, inter, intra, torsion, intra-best
    poses = v.poses(n_poses=n_modes)
    return np.array(energies), poses

def poses_to_rdkit(poses_pdbqt):
    """Docked PDBQT string -> RDKit mol (original atom order, one conformer per pose)."""
    pm = PDBQTMolecule(poses_pdbqt, skip_typing=True)
    return RDKitMolCreate.from_pdbqt_mol(pm)[0]

def heavy_rmsd(pose_mol, pose_id, ref_mol):
    """Symmetry-aware heavy-atom RMSD (no re-alignment) between a docked pose and the crystal pose."""
    a = Chem.RemoveHs(pose_mol); b = Chem.RemoveHs(ref_mol)
    return rdMolAlign.CalcRMS(a, b, prbId=pose_id, refId=-1)

def write_sdf(mol, path, conf_id=-1):
    w = Chem.SDWriter(path); w.write(mol, confId=conf_id); w.close()

# ---------------- interaction analysis (distance / angle based, no external tools) ----------------
import gemmi
AROM_RES = {"PHE": [["CG","CD1","CD2","CE1","CE2","CZ"]],
            "TYR": [["CG","CD1","CD2","CE1","CE2","CZ"]],
            "TRP": [["CG","CD1","CD2","NE1","CE2"], ["CD2","CE2","CE3","CZ2","CZ3","CH2"]],
            "HIS": [["CG","ND1","CD2","CE1","NE2"]]}
HYDROPHOBIC_RES = {"ALA","VAL","LEU","ILE","MET","PHE","TRP","PRO","TYR","CYS","LYS","ARG","THR"}

def load_receptor(pdb_path):
    st = gemmi.read_structure(pdb_path)
    rows = []
    for ch in st[0]:
        for res in ch:
            for a in res:
                rows.append(dict(chain=ch.name, resname=res.name, resnum=res.seqid.num, atom=a.name,
                                 el=a.element.name, xyz=np.array(a.pos.tolist())))
    df = pd.DataFrame(rows)
    df["label"] = df.resname + df.resnum.astype(str) + ":" + df.chain
    xyz = np.vstack(df.xyz.values)
    heavy = (df.el != "H").values
    # hydrogens attached to each heavy atom (within 1.3 A)
    hmap = {}
    hidx = np.where(~heavy)[0]; hv = np.where(heavy)[0]
    for h in hidx:
        d = np.linalg.norm(xyz[hv] - xyz[h], axis=1)
        j = hv[np.argmin(d)]
        if d.min() < 1.3:
            hmap.setdefault(j, []).append(h)
    return dict(df=df, xyz=xyz, heavy=heavy, hmap=hmap)

def _angle(a, b, c):   # angle at b (degrees)
    v1, v2 = a - b, c - b
    cosang = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-9)
    return float(np.degrees(np.arccos(np.clip(cosang, -1, 1))))

def analyze_pose(pose_mol, conf_id, rec, contact_cutoff=4.5, hb_dist=3.5, hb_angle=120.0,
                 hphob_dist=4.0, arom_dist=5.5):
    """Residue contacts for one docked pose. Labels are conservative: a polar contact is only called an
    H-bond when D...A <= hb_dist AND the D-H...A angle >= hb_angle; hydrophobic/aromatic are distance-only flags."""
    df, rxyz, heavy, hmap = rec["df"], rec["xyz"], rec["heavy"], rec["hmap"]
    conf = pose_mol.GetConformer(conf_id); lxyz = conf.GetPositions()
    lat = list(pose_mol.GetAtoms())
    rows = []
    rheavy_idx = np.where(heavy)[0]
    for a in lat:
        if a.GetAtomicNum() == 1:
            continue
        i = a.GetIdx(); sym = a.GetSymbol()
        d = np.linalg.norm(rxyz[rheavy_idx] - lxyz[i], axis=1)
        for k in np.where(d <= contact_cutoff)[0]:
            j = rheavy_idx[k]; r = df.iloc[j]; dist = float(d[k])
            itype, note = "close contact", f"heavy-atom distance <= {contact_cutoff} A"
            lig_polar, rec_polar = sym in ("N","O"), r.el in ("N","O")
            if lig_polar and rec_polar and dist <= hb_dist:
                best = None
                for nb in a.GetNeighbors():           # ligand donor hydrogens
                    if nb.GetAtomicNum() == 1:
                        ang = _angle(lxyz[i], lxyz[nb.GetIdx()], rxyz[j])
                        best = ang if best is None else max(best, ang)
                for h in hmap.get(j, []):             # receptor donor hydrogens
                    ang = _angle(rxyz[j], rxyz[h], lxyz[i])
                    best = ang if best is None else max(best, ang)
                if best is not None and best >= hb_angle:
                    itype, note = "hydrogen bond", f"D...A {dist:.2f} A, best D-H...A angle {best:.0f} deg"
                else:
                    itype, note = "polar contact", f"D...A {dist:.2f} A, H-bond angle criterion not met or no H available"
            elif sym == "C" and r.el == "C" and dist <= hphob_dist and r.resname in HYDROPHOBIC_RES \
                    and all(n.GetSymbol() in ("C","H") for n in a.GetNeighbors()):
                itype, note = "hydrophobic contact", f"C...C {dist:.2f} A (distance criterion only)"
            rows.append(dict(Residue=r.label, ResName=r.resname, ResNum=int(r.resnum), Chain=r.chain,
                             Receptor_atom=r.atom, Ligand_atom=f"{sym}{i}", Distance_A=round(dist, 2),
                             Interaction=itype, Note=note))
    # aromatic ring centroid check
    ri = pose_mol.GetRingInfo()
    lig_rings = [r for r in ri.AtomRings() if all(pose_mol.GetAtomWithIdx(x).GetIsAromatic() for x in r)]
    for res_label, sub in df.groupby("label"):
        rn = sub.resname.iloc[0]
        if rn not in AROM_RES:
            continue
        for names in AROM_RES[rn]:
            pts = sub[sub.atom.isin(names)]
            if len(pts) < len(names):
                continue
            rc = np.vstack(pts.xyz.values).mean(0)
            for ring in lig_rings:
                lc = lxyz[list(ring)].mean(0)
                dd = float(np.linalg.norm(rc - lc))
                if dd <= arom_dist:
                    rows.append(dict(Residue=res_label, ResName=rn, ResNum=int(sub.resnum.iloc[0]), Chain=sub.chain.iloc[0],
                                     Receptor_atom="ring", Ligand_atom="ring", Distance_A=round(dd, 2),
                                     Interaction="possible aromatic interaction",
                                     Note=f"ring centroid distance {dd:.2f} A (distance only, geometry not classified)"))
    out = pd.DataFrame(rows)
    if len(out):
        out = out.sort_values(["Residue", "Distance_A"]).reset_index(drop=True)
    return out

## CELL 4 - Upload the two input CSVs
Select **both** files in the upload dialog:
1. `master_docking_handover.csv` (Ayana's file with the ML-prioritised shortlists)
2. `descriptor_table.csv` (your Member 3 descriptor table)

In [ ]:
# CELL 4
require('D_IN')
from google.colab import files
up = files.upload()
for name, data in up.items():
    open(f"{D_IN}/{name}", "wb").write(data)
names = list(up.keys())
HANDOVER = next((n for n in names if "handover" in n.lower()), None)
DESC = next((n for n in names if "descriptor" in n.lower()), None)
assert HANDOVER and DESC, f"Need both files; got {names}"

ho = pd.read_csv(f"{D_IN}/{HANDOVER}")
desc = pd.read_csv(f"{D_IN}/{DESC}")
print("Handover columns:", list(ho.columns)); print("Handover rows:", len(ho))

tcols = [c for c in ho.columns if "target" in c.lower()]
assert tcols, "No 'target' column found in the handover file - screenshot the column list above and send it."
sel = ho[ho[tcols[0]].astype(str).str.contains("VEGFR", case=False, na=False)].copy()
if "Analog_ID" in sel.columns:
    sel = sel.drop_duplicates("Analog_ID")
print(f"VEGFR2 rows selected: {len(sel)} (expected 10)")
if len(sel) != 10:
    print("WARNING: not 10 rows - check the target column values:", ho[tcols[0]].unique())

# SMILES: use the handover SMILES, otherwise take it from the descriptor table
sm_col = next((c for c in sel.columns if c.lower() == "smiles"), None)
dcols = ["Analog_ID","SMILES","MW","LogP","TPSA","QED","SA_score","Lipinski_violations","PAINS_alerts","Brenk_alerts","NIH_alerts","Decision","Reason"]
dsub = desc[[c for c in dcols if c in desc.columns]].copy()
if sm_col is None:
    sel = sel.merge(dsub[["Analog_ID","SMILES"]], on="Analog_ID", how="left")
    sm_col = "SMILES"
sel = sel.merge(dsub.drop(columns=[c for c in ["SMILES"] if c in dsub.columns and sm_col == "SMILES" and "SMILES" in sel.columns], errors="ignore"),
                on="Analog_ID", how="left", suffixes=("_handover", ""))
sel.to_csv(f"{D_IN}/{TARGET}_shortlist.csv", index=False)
print(sel[["Analog_ID"]].assign(SMILES=sel[sm_col]).to_string(index=False))
print("saved", f"{D_IN}/{TARGET}_shortlist.csv")

## CELL 5 - Download 3VHE and check it is suitable (READ THE OUTPUT)
Prints protein identity, species, resolution, chains, engineered mutations, missing residues and all ligands, then picks the crystal ligand.
Send me a screenshot of this output if anything looks wrong (e.g. not human VEGFR2, no ligand, resolution very poor).

In [ ]:
# CELL 5
require('PDB_ID', 'D_REC', 'CHAIN')
import requests, gemmi, json
def fetch(url, path):
    r = requests.get(url, timeout=90); r.raise_for_status()
    open(path, "wb").write(r.content); return path
os.makedirs(f"{D_REC}/original", exist_ok=True)
pdb_path = fetch(f"https://files.rcsb.org/download/{PDB_ID}.pdb", f"{D_REC}/original/{PDB_ID}.pdb")
cif_path = fetch(f"https://files.rcsb.org/download/{PDB_ID}.cif", f"{D_REC}/original/{PDB_ID}.cif")

blk = gemmi.cif.read(cif_path).sole_block()
def val(tag):
    v = blk.find_value(tag)
    return gemmi.cif.as_string(v) if v is not None else "n/a"
summary = dict(PDB_ID=PDB_ID, title=val("_struct.title"), method=val("_exptl.method"),
               resolution=val("_refine.ls_d_res_high") if val("_refine.ls_d_res_high") != "n/a" else val("_reflns.d_resolution_high"))
print("Title      :", summary["title"]); print("Method     :", summary["method"]); print("Resolution :", summary["resolution"], "A")

print("\nEntities:")
for row in blk.find(["_entity.id", "_entity.type", "_entity.pdbx_description"]):
    print("  ", [gemmi.cif.as_string(x) for x in row])
print("Source organism:")
for tags in (["_entity_src_gen.entity_id", "_entity_src_gen.pdbx_gene_src_scientific_name", "_entity_src_gen.pdbx_gene_src_gene"],
             ["_entity_src_nat.entity_id", "_entity_src_nat.pdbx_organism_scientific"]):
    for row in blk.find(tags):
        print("  ", [gemmi.cif.as_string(x) for x in row])
print("Database reference (UniProt):")
for row in blk.find(["_struct_ref.entity_id", "_struct_ref.db_name", "_struct_ref.db_code", "_struct_ref.pdbx_db_accession"]):
    print("  ", [gemmi.cif.as_string(x) for x in row])
print("Sequence differences / engineered mutations:")
rows = list(blk.find(["_struct_ref_seq_dif.pdbx_pdb_strand_id", "_struct_ref_seq_dif.mon_id", "_struct_ref_seq_dif.pdbx_auth_seq_num",
                      "_struct_ref_seq_dif.db_mon_id", "_struct_ref_seq_dif.details"]))
for row in rows:
    print("  ", [gemmi.cif.as_string(x) for x in row])
if not rows: print("   none listed")

st = gemmi.read_structure(pdb_path); st.setup_entities()
print("\nChains and gaps in the observed residue numbering (missing residues):")
gaps_all = {}
for ch in st[0]:
    pol = ch.get_polymer()
    if len(pol) == 0: continue
    nums = [r.seqid.num for r in pol]
    gaps = [(a, b) for a, b in zip(nums, nums[1:]) if b - a > 1]
    gaps_all[ch.name] = gaps
    print(f"   chain {ch.name}: residues {nums[0]}-{nums[-1]}, {len(nums)} observed, gaps: {gaps if gaps else 'none'}")

print("\nNon-polymer / hetero residues (waters excluded):")
EXCL = {"HOH","GOL","EDO","SO4","PO4","PEG","DMS","ACT","CL","NA","K","BME","FMT","NO3","TRS","MPD","IOD","EPE","MES","CIT","TLA","PG4","1PE","NH4","BR","ACY","PGE","P6G","DTT","IPA","EOH"}
cands = []
for ch in st[0]:
    for r in ch:
        if r.is_water() or r.entity_type == gemmi.EntityType.Polymer: continue
        n_heavy = sum(1 for a in r if a.element.name != "H")
        cands.append(dict(chain=ch.name, resname=r.name, resnum=r.seqid.num, heavy_atoms=n_heavy))
cand_df = pd.DataFrame(cands); print(cand_df.to_string(index=False) if len(cand_df) else "none")

pool = cand_df[~cand_df.resname.isin(EXCL)] if len(cand_df) else cand_df
if LIG_RESNAME:
    pool = pool[pool.resname == LIG_RESNAME]
inchain = pool[pool.chain == CHAIN] if len(pool) else pool
pick = (inchain if len(inchain) else pool).sort_values("heavy_atoms", ascending=False)
assert len(pick), "No crystal ligand found - set LIG_RESNAME / CHAIN in CELL 2 or send me a screenshot."
LIG_RESNAME, LIG_CHAIN, LIG_SEQ = pick.iloc[0].resname, pick.iloc[0].chain, int(pick.iloc[0].resnum)
if LIG_CHAIN != CHAIN:
    print(f"NOTE: no ligand in chain {CHAIN}; using chain {LIG_CHAIN} instead"); CHAIN = LIG_CHAIN
summary.update(chain=CHAIN, ligand=LIG_RESNAME, ligand_resnum=LIG_SEQ, gaps=str(gaps_all.get(CHAIN)))
json.dump(summary, open(f"{D_REC}/structure_summary.json", "w"), indent=2)
print(f"\n>>> Using chain {CHAIN}, co-crystallised ligand {LIG_RESNAME} (residue {LIG_SEQ}) <<<")
print("Confirm: human VEGFR2 (KDR) kinase domain, reasonable resolution, ligand in the ATP pocket.")

## CELL 6 - Clean the receptor, extract the crystal ligand, define the docking box

In [ ]:
# CELL 6
require('pdb_path', 'LIG_RESNAME', 'CHAIN', 'D_REC', 'D_REF')
import gemmi
st = gemmi.read_structure(pdb_path); st.setup_entities()
model = st[0]
alt_atoms = sum(1 for ch in model for r in ch for a in r if a.altloc != "\0")

# crystal ligand -> its own PDB (before cleaning)
lig_res = next(r for r in model[LIG_CHAIN] if r.name == LIG_RESNAME and r.seqid.num == LIG_SEQ)
lig_st = gemmi.Structure(); lig_st.cell = st.cell; lig_st.spacegroup_hm = st.spacegroup_hm
lm = gemmi.Model("1"); lc = gemmi.Chain(LIG_CHAIN); lc.add_residue(lig_res); lm.add_chain(lc); lig_st.add_model(lm)
lig_st.remove_alternative_conformations(); lig_st.remove_hydrogens()
LIG_PDB = f"{D_REF}/{LIG_RESNAME}_crystal_ligand.pdb"
lig_st.write_pdb(LIG_PDB)
lig_xyz = np.array([a.pos.tolist() for r in lig_st[0][0] for a in r])

# receptor: selected chain, first alternate conformer, no H, no waters/ligands/ions (except KEEP_HET)
st.remove_alternative_conformations(); st.remove_hydrogens()
for name in [c.name for c in model if c.name != CHAIN]:
    model.remove_chain(name)
chain = model[CHAIN]
drop = [i for i, r in enumerate(chain) if r.is_water() or (r.entity_type != gemmi.EntityType.Polymer and r.name not in KEEP_HET)]
removed = [chain[i].name for i in drop]
for i in reversed(drop):
    del chain[i]
CLEAN_PDB = f"{D_REC}/{TARGET}_{PDB_ID}_chain{CHAIN}_clean.pdb"
st.write_pdb(CLEAN_PDB)

# docking box around the crystal ligand
CENTER = [round(float(x), 3) for x in lig_xyz.mean(0)]
extent = lig_xyz.max(0) - lig_xyz.min(0)
SIZE = [round(float(max(e + BOX_EXTRA, BOX_MIN)), 1) for e in extent]
print("residues kept:", len(chain), "| altloc atoms resolved (first conformer kept):", alt_atoms)
print("removed hetero/water residues:", pd.Series(removed).value_counts().to_dict())
print("ligand heavy atoms:", len(lig_xyz), "| ligand extent (A):", np.round(extent, 1))
print("BOX CENTER:", CENTER, "| BOX SIZE:", SIZE)
json.dump(dict(center=CENTER, size=SIZE, ligand=LIG_RESNAME, chain=CHAIN), open(f"{D_REC}/docking_box.json", "w"), indent=2)

## CELL 7 - Add hydrogens (pH 7.4) and prepare the receptor PDBQT with Meeko

In [ ]:
# CELL 7
require('CLEAN_PDB', 'CENTER', 'SIZE', 'D_REC')
import subprocess, shutil
from pdbfixer import PDBFixer
from openmm.app import PDBFile
fx = PDBFixer(filename=CLEAN_PDB)
fx.findMissingResidues(); fx.missingResidues = {}          # do not build missing loops
fx.findNonstandardResidues(); fx.replaceNonstandardResidues()
fx.findMissingAtoms(); fx.addMissingAtoms()
fx.addMissingHydrogens(7.4)
PROT_H_PDB = f"{D_REC}/{TARGET}_{PDB_ID}_chain{CHAIN}_H.pdb"
PDBFile.writeFile(fx.topology, fx.positions, open(PROT_H_PDB, "w"), keepIds=True)

REC_BASE = f"{D_REC}/receptor"
cmd = ["mk_prepare_receptor.py", "--read_pdb", PROT_H_PDB, "-o", REC_BASE, "-p", "-v",
       "--box_center", *map(str, CENTER), "--box_size", *map(str, SIZE)]
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout[-800:]); 
if res.returncode != 0:
    print(res.stderr[-1500:]); raise RuntimeError("Meeko receptor preparation failed - screenshot the message above and send it to me.")
REC_PDBQT = f"{REC_BASE}.pdbqt"
assert os.path.exists(REC_PDBQT), "receptor.pdbqt was not created"
print("Receptor ready:", REC_PDBQT)

## CELL 8 - Reference-ligand redocking validation (important)
Docks the co-crystallised ligand (fresh 3D conformer, not the crystal pose) back into the receptor and compares with the crystal pose.
Pass = Model 1 heavy-atom RMSD below about 2 A.

In [ ]:
# CELL 8
require('REC_PDBQT', 'CENTER', 'SIZE', 'LIG_PDB', 'LIG_RESNAME')
import requests, gemmi
# official chemical component -> SMILES with correct bond orders / stereochemistry
cc = requests.get(f"https://files.rcsb.org/ligands/view/{LIG_RESNAME}.cif", timeout=60); cc.raise_for_status()
cblk = gemmi.cif.read_string(cc.text)[-1]
smi = None
for typ_pref, prog_pref in [("SMILES_CANONICAL", "OpenEye"), ("SMILES", "OpenEye"), ("SMILES_CANONICAL", ""), ("SMILES", "")]:
    for row in cblk.find(["_pdbx_chem_comp_descriptor.type", "_pdbx_chem_comp_descriptor.program", "_pdbx_chem_comp_descriptor.descriptor"]):
        t, p, d = [gemmi.cif.as_string(x) for x in row]
        if t == typ_pref and prog_pref in p:
            smi = d; break
    if smi: break
assert smi, "No SMILES found for the reference ligand component"
print("Reference ligand", LIG_RESNAME, "SMILES:", smi)
template = Chem.MolFromSmiles(smi)

crystal = Chem.MolFromPDBFile(LIG_PDB, removeHs=True, proximityBonding=True)
try:
    ref = AllChem.AssignBondOrdersFromTemplate(template, crystal)
except Exception as e:
    raise RuntimeError(f"Could not map the crystal ligand to its chemical definition ({e}). "
                       "Check for missing atoms / alternate conformations in the ligand and send me a screenshot.")
Chem.AssignStereochemistryFrom3D(ref)
write_sdf(ref, f"{D_REF}/{LIG_RESNAME}_reference_crystal_pose.sdf")

probe = Chem.AddHs(template)
ps = AllChem.ETKDGv3(); ps.randomSeed = SEED; ps.useRandomCoords = True
assert AllChem.EmbedMolecule(probe, ps) == 0, "could not embed the reference ligand"
AllChem.MMFFOptimizeMolecule(probe, maxIters=5000)
E, P = run_vina(mol_to_pdbqt(probe), REC_PDBQT, CENTER, SIZE, EXHAUSTIVENESS, N_MODES, SEED)
open(f"{D_REF}/{LIG_RESNAME}_redocked.pdbqt", "w").write(P)
pm = poses_to_rdkit(P)
write_sdf(pm, f"{D_REF}/{LIG_RESNAME}_redocked_model1.sdf", 0)
rmsds = [heavy_rmsd(pm, k, ref) for k in range(pm.GetNumConformers())]
VALIDATION = dict(ligand=LIG_RESNAME, rmsd_model1=round(rmsds[0], 2), rmsd_best_of_modes=round(min(rmsds), 2),
                  best_mode=int(np.argmin(rmsds)) + 1, vina_model1=float(E[0][0]), cutoff=RMSD_CUTOFF,
                  passed=bool(rmsds[0] < RMSD_CUTOFF))
json.dump(VALIDATION, open(f"{D_REF}/redocking_validation.json", "w"), indent=2)
print("RMSD per mode (A):", [round(x, 2) for x in rmsds])
print(VALIDATION)
if VALIDATION["passed"]:
    print("\nVALIDATION PASSED - continue.")
else:
    print("\nVALIDATION NOT PASSED (Model 1 RMSD >= 2 A). Do not continue yet: check receptor chain, ligand preparation, "
          "protonation, box and atom mapping. Send me a screenshot of this output.")

## CELL 9 - Prepare the 10 shortlisted ligands (RDKit ETKDG + MMFF, then Meeko PDBQT)

In [ ]:
# CELL 9
require('sel', 'sm_col', 'D_LIG', 'VALIDATION')
if not VALIDATION["passed"] and not FORCE_CONTINUE:
    raise RuntimeError("Redocking validation did not pass (CELL 8). Fix it first, or set FORCE_CONTINUE = True in CELL 2 after investigating.")
os.makedirs(f"{D_LIG}/sdf", exist_ok=True); os.makedirs(f"{D_LIG}/pdbqt", exist_ok=True)
LIGS, failed = {}, []
for _, r in sel.iterrows():
    aid, smi = r["Analog_ID"], r[sm_col]
    try:
        mol, pq, mm = prepare_ligand(smi, SEED)
        mol.SetProp("_Name", aid); write_sdf(mol, f"{D_LIG}/sdf/{aid}.sdf")
        open(f"{D_LIG}/pdbqt/{aid}.pdbqt", "w").write(pq)
        LIGS[aid] = pq
        print(f"{aid}: OK (MMFF status {mm}: 0=converged)")
    except Exception as e:
        failed.append(dict(Analog_ID=aid, SMILES=smi, error=str(e))); print(f"{aid}: FAILED - {e}")
pd.DataFrame(failed).to_csv(f"{D_LIG}/failed_preparations.csv", index=False)
print(f"\nPrepared {len(LIGS)} of {len(sel)} ligands. Failures: {len(failed)}")

## CELL 10 - Dock all shortlisted ligands (this is the slow step: roughly 10-30 minutes on free Colab)

In [ ]:
# CELL 10
require('LIGS', 'REC_PDBQT', 'CENTER', 'SIZE', 'D_OUT')
import time
POSES, rows = {}, []
for n, (aid, pq) in enumerate(LIGS.items(), 1):
    t0 = time.time()
    E, P = run_vina(pq, REC_PDBQT, CENTER, SIZE, EXHAUSTIVENESS, N_MODES, SEED)
    open(f"{D_OUT}/{aid}_docked.pdbqt", "w").write(P)
    POSES[aid] = poses_to_rdkit(P)
    rows.append(dict(Analog_ID=aid, Vina_best_kcal_mol=float(E[0][0]), Vina_all_modes=";".join(f"{e[0]:.2f}" for e in E), n_modes=len(E)))
    print(f"[{n}/{len(LIGS)}] {aid}: best {E[0][0]:.2f} kcal/mol ({time.time()-t0:.0f}s)")
dock = pd.DataFrame(rows).sort_values("Vina_best_kcal_mol")
dock.to_csv(f"{D_OUT}/docking_results.csv", index=False)
print(dock.to_string(index=False))

## CELL 11 - Rank the candidates (never docking score alone)
Combines Vina score, ML predicted pIC50, RF/XGBoost agreement, QED, SA score and PAINS/Brenk alerts.
The cell prints which columns it detected - check that they are the right ones.

In [ ]:
# CELL 11
require('dock', 'sel', 'sm_col')
import re
cols = list(sel.columns)
def pick(*pats):
    for p in pats:
        for c in cols:
            if re.search(p, c, re.I): return c
RF_COL  = pick(r"rf.*pic50", r"pic50.*rf", r"random.?forest", r"^rf")
XGB_COL = pick(r"xgb.*pic50", r"pic50.*xgb", r"xgboost", r"^xgb")
ML_COL  = pick(r"(mean|ensemble|avg|average|consensus).*pic50", r"pic50.*(mean|ensemble|avg|average|consensus)")
print("Detected -> RF:", RF_COL, "| XGB:", XGB_COL, "| ML mean:", ML_COL)
res = sel.merge(dock, on="Analog_ID", how="inner")
if ML_COL is None and RF_COL and XGB_COL:
    res["ML_pIC50"] = res[[RF_COL, XGB_COL]].mean(axis=1)
elif ML_COL:
    res["ML_pIC50"] = res[ML_COL]
else:
    raise RuntimeError("Could not detect ML pIC50 columns. Screenshot the column list from CELL 4 and send it to me.")
res["RF_XGB_abs_diff"] = (res[RF_COL] - res[XGB_COL]).abs() if RF_COL and XGB_COL else np.nan
for c in ["QED", "SA_score"]:
    assert c in res.columns, f"{c} missing - was descriptor_table.csv uploaded in CELL 4?"
res["PAINS_flag"] = res["PAINS_alerts"].fillna("").astype(str).str.len() > 0
res["Brenk_flag"] = res["Brenk_alerts"].fillna("").astype(str).str.len() > 0
rk = pd.DataFrame({"r_vina": res["Vina_best_kcal_mol"].rank(ascending=True, method="min"),
                   "r_ml":   res["ML_pIC50"].rank(ascending=False, method="min"),
                   "r_qed":  res["QED"].rank(ascending=False, method="min"),
                   "r_sa":   res["SA_score"].rank(ascending=True, method="min")})
res["Composite_rank_score"] = rk.mean(axis=1)          # lower = better
res = res.sort_values(["Composite_rank_score", "Vina_best_kcal_mol"]).reset_index(drop=True)
res["Composite_rank"] = np.arange(1, len(res) + 1)
show = ["Composite_rank", "Analog_ID", "Vina_best_kcal_mol", "ML_pIC50", "RF_XGB_abs_diff", "QED", "SA_score", "PAINS_flag", "Brenk_flag"]
print(res[show].round(2).to_string(index=False))
res.to_csv(f"{D_FIN}/candidate_ranking_all10.csv", index=False)

## CELL 12 - Interaction analysis and 3D views for the top 3 candidates (Model 1 pose)
Hydrogen bonds are only labelled when donor-acceptor distance <= 3.5 A **and** the D-H...A angle >= 120 degrees.
Hydrophobic and aromatic entries are distance-only flags; everything else is a plain close contact.

In [ ]:
# CELL 12
require('res', 'POSES', 'PROT_H_PDB', 'D_AN')
import py3Dmol
rec = load_receptor(PROT_H_PDB)
TOP3 = list(res.head(3)["Analog_ID"])
all_int, summ = [], {}
for aid in TOP3:
    pm = POSES[aid]
    write_sdf(pm, f"{D_AN}/{aid}_model1.sdf", 0)
    inter = analyze_pose(pm, 0, rec)
    inter.insert(0, "Analog_ID", aid); inter.insert(1, "Pose", "Model 1")
    inter.to_csv(f"{D_AN}/{aid}_interactions.csv", index=False)
    all_int.append(inter)
    hb = inter[inter.Interaction == "hydrogen bond"]
    summ[aid] = dict(n_hbond=len(hb), hbond_residues=", ".join(sorted(set(hb.Residue))),
                     n_hydrophobic_res=inter[inter.Interaction == "hydrophobic contact"].Residue.nunique(),
                     n_contact_res=inter.Residue.nunique())
    print(f"\n=== {aid} ===  H-bonds: {len(hb)}  | hydrophobic-contact residues: {summ[aid]['n_hydrophobic_res']}  | residues within 4.5 A: {summ[aid]['n_contact_res']}")
    print(inter[inter.Interaction != "close contact"][["Residue", "Receptor_atom", "Ligand_atom", "Distance_A", "Interaction"]].to_string(index=False))
    near = sorted(set(inter.ResNum))
    v = py3Dmol.view(width=820, height=560)
    v.addModel(open(PROT_H_PDB).read(), "pdb"); v.setStyle({"model": 0}, {"cartoon": {"color": "lightgrey"}})
    v.addStyle({"model": 0, "resi": near}, {"stick": {"colorscheme": "cyanCarbon", "radius": 0.15}})
    v.addModel(Chem.MolToMolBlock(pm, confId=0), "sdf"); v.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon", "radius": 0.25}})
    v.zoomTo({"model": 1})
    try: v.write_html(f"{D_AN}/{aid}_pose_view.html")
    except Exception as e: print("could not save html view:", e)
    v.show()
pd.concat(all_int).to_csv(f"{D_AN}/residue_contacts_interactions.csv", index=False)
print("\nsaved", f"{D_AN}/residue_contacts_interactions.csv")

## CELL 13 - Final candidates, report and download

In [ ]:
# CELL 13
require('res', 'summ', 'VALIDATION', 'D_FIN')
import shutil
fin = res.copy()
fin["Interaction_Hbonds_Model1"] = fin["Analog_ID"].map(lambda a: summ.get(a, {}).get("n_hbond"))
fin["Hbond_residues_Model1"] = fin["Analog_ID"].map(lambda a: summ.get(a, {}).get("hbond_residues"))
fin["Selected_for_further_work"] = (fin["Composite_rank"] <= 3) & (~fin["PAINS_flag"])
fin["Note"] = "Computational prediction only; ranked on docking + ML + drug-likeness + SA, not docking alone"
fin.to_csv(f"{D_FIN}/final_candidate_ranking.csv", index=False)
chosen = fin[fin.Selected_for_further_work]
print(fin[["Composite_rank", "Analog_ID", "Vina_best_kcal_mol", "ML_pIC50", "QED", "SA_score", "PAINS_flag", "Interaction_Hbonds_Model1", "Selected_for_further_work"]].round(2).to_string(index=False))

def md_table(df):
    return "\n".join(["| " + " | ".join(df.columns) + " |", "|" + "---|" * len(df.columns)] +
                      ["| " + " | ".join(str(x) for x in r) + " |" for r in df.values.tolist()])
summ_json = json.load(open(f"{D_REC}/structure_summary.json"))
rep = f'''# {TARGET} docking report (PDB {PDB_ID}) - Abhi

## Methods
- Receptor: PDB {PDB_ID} ({summ_json["title"]}), resolution {summ_json["resolution"]} A, chain {CHAIN}; crystal ligand {LIG_RESNAME}.
  First alternate conformer kept; waters, ligands and unrelated hetero atoms removed; hydrogens added at pH 7.4 (PDBFixer); missing loops not rebuilt; PDBQT by Meeko.
- Docking box (from crystal ligand centroid): centre {CENTER}, size {SIZE} A.
- Ligands: 10 ML-shortlisted analogs for {TARGET}; RDKit ETKDGv3 + MMFF optimisation; Meeko PDBQT.
- AutoDock Vina 1.2.7 (Vina scoring), exhaustiveness {EXHAUSTIVENESS}, {N_MODES} modes, seed {SEED}.

## Redocking validation
{VALIDATION}

## Docking and ranking (lower composite score = better)
{md_table(fin[["Composite_rank","Analog_ID","Vina_best_kcal_mol","ML_pIC50","QED","SA_score","PAINS_flag","Brenk_flag"]].round(2))}

Selected for further work: {", ".join(chosen.Analog_ID) if len(chosen) else "none"}

## Interaction analysis (top 3, Model 1)
H-bond = donor...acceptor <= 3.5 A and D-H...A angle >= 120 deg. Hydrophobic/aromatic entries are distance-based flags only.
{pd.DataFrame(summ).T.reset_index().rename(columns={"index":"Analog_ID"}).to_string(index=False)}

## Limitations
Vina scores are approximate and are not comparable between different targets. Ligand protonation/tautomer states were not enumerated.
All results are computational predictions and are not experimentally confirmed activity.
'''
open(f"{D_FIN}/methodology_and_results_report.md", "w").write(rep)
shutil.make_archive(BASE, "zip", BASE)
from google.colab import files
files.download(f"{BASE}.zip")
print("Done. Downloaded", f"{BASE}.zip")